[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/salas/S1_que_ajusta.ipynb)

# Sala: ¿qué ajusta mejor?

10 minutos, en salas de 3 o 4. No se califica. Repártanse los papeles: **quien lee** cada paso en voz alta,
**quien corre** las celdas compartiendo pantalla, **quien duda** ("¿y esto qué quiere decir?") y **quien escribe** en el chat al volver.
Corran las celdas en orden con Shift + Enter. Las dos primeras preparan el entorno hasta que aparezca `✅ entorno listo`.

Las tres varas para medir un modelo: **MAE** (el error típico), **RMSE** (igual, pero los errores grandes pesan más)
y **R²** (qué tanto mejora a "decir siempre el promedio": 0 es igual que el promedio, 1 es perfecto).

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dammg_utils import COLORES
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
du.estilo_dammg()
np.set_printoptions(precision=4, suppress=True)

In [ ]:
# Tres datos: 400 horas de MiBici, autos (potencia y rendimiento) y un restaurante (cuenta y propina)
def externo(nombre):
    """Copia local del repo si existe; si no, el CSV público de seaborn-data."""
    import pathlib
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        if (p / "data" / "externos" / nombre).is_file():
            return pd.read_csv(p / "data" / "externos" / nombre)
    return pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/" + nombre)


df = pd.read_parquet(RUTA_DATOS)
mibici = df[df.particion == "train"].sample(400, random_state=0)
autos = externo("mpg.csv").dropna(subset=["horsepower"])
restaurante = externo("tips.csv")

DATOS = {
    "A. MiBici": (mibici.departures_lag_168h, mibici.departures, "salidas hace una semana", "salidas hoy"),
    "B. Autos": (autos.horsepower, autos.mpg, "caballos de fuerza", "millas por galón"),
    "C. Restaurante": (restaurante.total_bill, restaurante.tip, "cuenta (dólares)", "propina (dólares)"),
}


def ajusta(x, y, grado=1):
    """Ajusta un polinomio del grado pedido (grado 1 es una recta) y regresa el modelo."""
    X = np.asarray(x, float)[:, None]
    return make_pipeline(PolynomialFeatures(grado), LinearRegression()).fit(X, y)


def varas(x, y, modelo):
    y_hat = modelo.predict(np.asarray(x, float)[:, None])
    return {"MAE": mean_absolute_error(y, y_hat), "RMSE": root_mean_squared_error(y, y_hat), "R²": r2_score(y, y_hat)}


def dibuja(ax, x, y, modelo, titulo, ex="x", ey="y"):
    g = np.linspace(np.min(x), np.max(x), 200)[:, None]
    ax.scatter(x, y, s=22, color=COLORES["datos"], edgecolor=COLORES["tinta"], lw=0.7)
    ax.plot(g[:, 0], modelo.predict(g), color=COLORES["error"], lw=3)
    ax.set(title=titulo, xlabel=ex, ylabel=ey)


print({k: len(v[0]) for k, v in DATOS.items()})

## 1. Tres nubes, una recta en cada una

A cada nube de puntos le ajustamos su mejor recta.

**¿En cuál la recta NO sigue la forma de los puntos?**

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (nombre, (x, y, ex, ey)) in zip(axs, DATOS.items()):
    dibuja(ax, x, y, ajusta(x, y), nombre, ex, ey)
fig.tight_layout()
plt.show()

**¿Qué esperamos ver?** En MiBici una nube ancha que sube; en el restaurante una nube que sube más o menos derecha.
En los autos los puntos bajan como curva y la recta se les despega en las orillas.

Ahora las tres varas de cada recta.

In [ ]:
tabla = pd.DataFrame({nombre: varas(x, y, ajusta(x, y)) for nombre, (x, y, _, _) in DATOS.items()}).T
tabla.round(2)

**¿Qué esperamos ver?** Las varas de cada dato vienen en sus propias unidades (bicis, millas por galón, dólares),
así que el MAE y el RMSE no se comparan entre filas. El R² sí: va de 0 a 1 en los tres.

**¿El R² más alto es el que mejor se ve?**

## 2. Con una curva

Los autos otra vez, ahora la recta (grado 1) junto a una curva de grado 2.

**¿Se ve mejor? ¿Bajaron los errores?**

In [ ]:
x, y, ex, ey = DATOS["B. Autos"]
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, grado in zip(axs, (1, 2)):
    m = ajusta(x, y, grado)
    v = varas(x, y, m)
    dibuja(ax, x, y, m, f"Grado {grado}: MAE {v['MAE']:.2f}  RMSE {v['RMSE']:.2f}  R² {v['R²']:.2f}", ex, ey)
fig.tight_layout()
plt.show()

**¿Qué esperamos ver?** La curva de grado 2 se dobla con los puntos y las tres varas mejoran: MAE y RMSE bajan, R² sube.

Muevan el grado de 1 a 6 y fíjense en la curva y en los números.

In [ ]:
def ver_grado(grado=1):
    m = ajusta(x, y, grado)
    v = varas(x, y, m)
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    dibuja(ax, x, y, m, f"Grado {grado}: MAE {v['MAE']:.2f}  RMSE {v['RMSE']:.2f}  R² {v['R²']:.2f}", ex, ey)
    plt.show()


try:
    import ipywidgets as widgets
    widgets.interact(ver_grado, grado=widgets.IntSlider(value=1, min=1, max=6, step=1, description="grado"))
except ImportError:
    print("No hay ipywidgets: usa la celda de abajo.")

In [ ]:
# Si el slider no aparece: cambien el grado (1 a 6) y vuelvan a correr esta celda
ver_grado(2)

**¿Qué esperamos ver?** El salto grande es de 1 a 2. De ahí en adelante los números casi no se mueven
y con grados altos la curva empieza a hacer cosas raras en las orillas, donde hay pocos autos.

## 3. El cuarteto de Anscombe

Cuatro datos de 11 puntos cada uno, inventados en 1973 por Frank Anscombe, cada uno con su mejor recta.

**Mismos números: ¿se ven igual? ¿En cuál usarían una recta?**

In [ ]:
ax_ = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
ANSCOMBE = {
    "I": (ax_, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    "II": (ax_, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    "III": (ax_, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    "IV": ([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}

fig, axs = plt.subplots(2, 2, figsize=(10, 7.5), sharex=True, sharey=True)
filas = {}
for ax, (nombre, (xa, ya)) in zip(axs.flat, ANSCOMBE.items()):
    m = ajusta(xa, ya)
    v = varas(xa, ya, m)
    filas[nombre] = v
    dibuja(ax, np.array(xa), np.array(ya), m, f"Anscombe {nombre}: R² {v['R²']:.2f}  RMSE {v['RMSE']:.2f}")
fig.tight_layout()
plt.show()
pd.DataFrame(filas).T.round(2)

**¿Qué esperamos ver?** La tabla da casi lo mismo en los cuatro: R² 0.67 y RMSE 1.12. Las gráficas no se parecen:
uno es una nube que sube, otro una curva, otro una línea con un punto fuera y otro una columna con un solo punto lejos.

## Al volver a la sala principal

Quien escribe prepara esta línea y la manda al chat cuando el profesor diga "ya". Envíenla al mismo tiempo para comparar ideas que salieron por separado.

`la recta no sigue la forma en: __ / con la curva, los errores: __ / Anscombe con recta buena: __`

| Espacio | Qué va | De dónde sale |
|---|---|---|
| la recta no sigue la forma en | A, B o C | Paso 1 |
| con la curva, los errores | Si bajaron o subieron, y cuánto | Paso 2 |
| Anscombe con recta buena | I, II, III o IV (puede ser más de uno) | Paso 3 |

No tiene que estar todo bien ni completo. Si no llegaron a un paso, pongan `?` y manden la línea igual.